# GEPA Medical Condition Classification

Use the `sv_medical_200.csv` dataset, take `patient_notes` as input, and predict the gold medical condition in `label`.
This notebook follows the same staged structure as `gepa_3_tldr.ipynb`, but uses exact-match classification accuracy as the primary optimization target.

In [ ]:
# ============================================================
# 0) IMPORTS & CONFIG
# ============================================================
import csv
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import dspy


def find_repo_root() -> Path:
def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "sv_medical_200.csv"

# Base model for reasoning
MODEL_NAME = os.getenv("DSPY_MODEL", "openai/openai/gpt-oss-20b")
API_BASE = os.getenv("DSPY_API_BASE", "http://10.0.10.51:8123/v1")
API_KEY = os.getenv("DSPY_API_KEY", "sv-openai-api-key")


lm=dspy.LM(
    # 'openai/gpt-4o-mini'
    "openai/openai/gpt-oss-20b",
    api_base="http://10.0.10.51:8124/v1",
    api_key="sv-openai-api-key",
    
)

dspy.configure(lm=lm)

# Stronger / more exploratory LM for reflection
reflection_lm = dspy.LM(
    # 'openai/gpt-4o-mini'
    "openai/openai/gpt-oss-20b",
    api_base="http://10.0.10.51:8124/v1",
    api_key="sv-openai-api-key",
)

print(f"Repo root: {REPO_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Model: {MODEL_NAME}")

Repo root: /home/skumaran/repositories/ai_agents_bootcamp/dspy_poc
Dataset path: /home/skumaran/repositories/ai_agents_bootcamp/dspy_poc/data/sv_medical_200.csv
Model: openai/openai/gpt-oss-20b


In [2]:
# ============================================================
# 1) LOAD DATASET + STRATIFIED SPLITS
# ============================================================
def load_medical_rows(path: Path):
    with path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))

    cleaned = []
    for row in rows:
        patient_notes = row["patient_notes"].strip()
        label = row["label"].strip()
        reasoning = row.get("reasoning", "").strip()
        if patient_notes and label:
            cleaned.append(
                {
                    "patient_notes": patient_notes,
                    "label": label,
                    "reasoning": reasoning,
                }
            )
    return cleaned


def stratified_split(rows, train_ratio=0.70, val_ratio=0.15, seed=7):
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for row in rows:
        buckets[row["label"]].append(row)

    train_rows, val_rows, test_rows = [], [], []
    for label, items in sorted(buckets.items()):
        items = items[:]
        rng.shuffle(items)

        n_total = len(items)
        n_train = max(1, int(round(n_total * train_ratio)))
        n_val = max(1, int(round(n_total * val_ratio)))

        if n_train + n_val >= n_total:
            n_val = max(1, n_total - n_train - 1)
        n_test = n_total - n_train - n_val
        if n_test <= 0:
            n_test = 1
            if n_train > n_val:
                n_train -= 1
            else:
                n_val -= 1

        train_rows.extend(items[:n_train])
        val_rows.extend(items[n_train:n_train + n_val])
        test_rows.extend(items[n_train + n_val:])

    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    rng.shuffle(test_rows)
    return train_rows, val_rows, test_rows


def to_examples(rows):
    return [
        dspy.Example(
            {
                "patient_notes": row["patient_notes"],
                "gold_label": row["label"],
                "reasoning": row["reasoning"],
            }
        ).with_inputs("patient_notes")
        for row in rows
    ]


rows = load_medical_rows(DATA_PATH)
train_rows, val_rows, test_rows = stratified_split(rows)

train_set = to_examples(train_rows)
val_set = to_examples(val_rows)
test_set = to_examples(test_rows)

label_counts = Counter(row["label"] for row in rows)
label_space = sorted(label_counts)

print(f"Total rows: {len(rows)}")
print(f"Train / Val / Test: {len(train_set)} / {len(val_set)} / {len(test_set)}")
print("\nLabel distribution:")
for label, count in sorted(label_counts.items()):
    print(f"- {label}: {count}")

print("\nAllowed labels:")
print(label_space)

Total rows: 202
Train / Val / Test: 142 / 30 / 30

Label distribution:
- Arthritis: 34
- COPD/Respiratory: 33
- Cardiovascular: 34
- Diabetes: 33
- Eye Condition: 34
- Neurological: 34

Allowed labels:
['Arthritis', 'COPD/Respiratory', 'Cardiovascular', 'Diabetes', 'Eye Condition', 'Neurological']


In [3]:
# ============================================================
# 2) SIGNATURE + MODULE
# ============================================================
ALLOWED_LABELS_TEXT = ", ".join(label_space)


class PredictMedicalCondition(dspy.Signature):
    """
    Read the patient notes and predict the single best medical condition label.
    """

    patient_notes: str = dspy.InputField(desc="Patient history, symptoms, and clinical notes")
    medical_condition: str = dspy.OutputField(desc="One medical condition")


class MedicalConditionModule(dspy.Module):
    def __init__(self):
        self.generator = dspy.ChainOfThought(PredictMedicalCondition)

    def forward(self, patient_notes: str):
        out = self.generator(patient_notes=patient_notes)
        return dspy.Prediction(medical_condition=out.medical_condition)


program = MedicalConditionModule()
print(ALLOWED_LABELS_TEXT)

Arthritis, COPD/Respiratory, Cardiovascular, Diabetes, Eye Condition, Neurological


In [4]:
# ============================================================
# 3) HELPERS (label normalization, reporting, evaluation)
# ============================================================
CANONICAL_LABELS = {re.sub(r'[^a-z0-9]+', '', label.lower()): label for label in label_space}


def normalize_label(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", (text or "").strip().lower())


def canonicalize_label(text: str) -> str:
    return CANONICAL_LABELS.get(normalize_label(text), (text or "").strip())


def is_valid_label(text: str) -> bool:
    return normalize_label(text) in CANONICAL_LABELS


def exact_match_score(gold: str, pred: str) -> float:
    return 1.0 if normalize_label(gold) == normalize_label(pred) else 0.0


def dataset_accuracy(module, dataset):
    correct = 0
    total = len(dataset)
    rows = []

    for example in dataset:
        pred = module(patient_notes=example["patient_notes"])
        pred_label = canonicalize_label(pred.medical_condition)
        gold_label = example["gold_label"]
        score = exact_match_score(gold_label, pred_label)
        correct += score
        rows.append(
            {
                "gold": gold_label,
                "pred": pred_label,
                "raw_pred": pred.medical_condition,
                "correct": bool(score),
            }
        )

    accuracy = correct / total if total else 0.0
    return accuracy, rows


def print_sample_predictions(results, n=5):
    for i, row in enumerate(results[:n], start=1):
        print(f"Example {i}: gold={row['gold']} | pred={row['pred']} | correct={row['correct']}")
        if row["raw_pred"] != row["pred"]:
            print(f"  raw_pred={row['raw_pred']}")

In [5]:
# ============================================================
# 4) FEEDBACK FUNCTIONS — OPTIMIZED FOR CLASSIFICATION ACCURACY
# ============================================================
def feedback_valid_label(pred_label: str):
    valid = is_valid_label(pred_label)
    score = 1.0 if valid else 0.0

    if valid:
        fb = f"You predicted a valid label: {canonicalize_label(pred_label)}."
    else:
        fb = (
            "Your output is not one of the allowed labels. "
            f"Choose exactly one from: {ALLOWED_LABELS_TEXT}."
        )
    return fb, score


def feedback_accuracy(example, pred_label: str):
    gold_label = example["gold_label"]
    pred_canonical = canonicalize_label(pred_label)
    score = exact_match_score(gold_label, pred_canonical)

    if score == 1.0:
        fb = f"Correct. The predicted label matches the gold label: {gold_label}."
    else:
        fb = (
            f"Incorrect. You predicted '{pred_canonical}' but the gold label is '{gold_label}'. "
            "Pay closer attention to symptom patterns, organ system clues, and disease-specific terminology in the notes."
        )
    return fb, score


In [6]:
# ============================================================
# 5) METRIC (SCALAR ONLY)
# ============================================================
def metric(example, pred, trace=None, pred_name=None, pred_trace=None):
    _, s_valid = feedback_valid_label(pred.medical_condition)
    _, s_acc = feedback_accuracy(example, pred.medical_condition)

    total = (0.10 * s_valid) + (0.90 * s_acc)
    return total


In [7]:
# ============================================================
# BASELINE EVALUATION
# ============================================================
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=8,
    display_table=True,
    display_progress=True,
)

print("\n== Baseline Evaluation ==")
evaluate(program)

baseline_accuracy, baseline_rows = dataset_accuracy(program, test_set)
print(f"\nBaseline exact-match accuracy: {baseline_accuracy:.2%}")
print_sample_predictions(baseline_rows)


== Baseline Evaluation ==
Average Metric: 0.00 / 30 (0.0%): 100%|██████████| 30/30 [00:16<00:00,  1.77it/s]

2026/04/01 21:10:01 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 30 (0.0%)


,patient_notes,gold_label,reasoning,medical_condition,metric
0,12-year-old female whose parents noticed progressive pallor of her...,Eye Condition,The presence of pallor of the optic discs and severely reduced vis...,Multiple Sclerosis,✔️ [0.000]
1,"A 78-year-old female, recently discharged from the hospital after ...",COPD/Respiratory,The patient's recent stroke and dysphagia are significant risk fac...,Aspiration pneumonia,✔️ [0.000]
2,19-year-old female presents after an episode of syncope while swim...,Cardiovascular,The occurrence of syncope during exertion (swimming) in a young in...,Long QT syndrome,✔️ [0.000]
3,A 75-year-old male with a history of chronic kidney disease and di...,Arthritis,While the polyarticular involvement might initially suggest rheuma...,Acute gout (acute gouty arthritis),✔️ [0.000]
4,"81-year-old female admitted after a fall at home, sustaining a hip...",Cardiovascular,"The patient's recurrent lightheadedness, dizziness, and syncope up...",Orthostatic hypotension,✔️ [0.000]
5,"An 85-year-old male, residing in a skilled nursing facility, prese...",COPD/Respiratory,"The patient's advanced age, institutionalization, acute onset of f...",Community-acquired pneumonia,✔️ [0.000]
6,A 58-year-old obese male (BMI 38 kg/m²) presents with chronic dayt...,COPD/Respiratory,While ankle edema and dyspnea might suggest primary cardiac or ren...,Obstructive sleep apnea,✔️ [0.000]
7,92-year-old female found unresponsive at home. Paramedics report p...,Cardiovascular,"The presentation of sudden collapse, profound hypotension, a pulsa...",Ruptured abdominal aortic aneurysm,✔️ [0.000]
8,"A 72-year-old male, with a long-standing history of tobacco abuse ...",COPD/Respiratory,"The patient's advanced age, extensive smoking history, chronic pro...",Acute exacerbation of chronic obstructive pulmonary disease,✔️ [0.000]
9,A 9-month-old male infant presents with a 3-day history of rhinorr...,COPD/Respiratory,"The patient's age, viral prodrome, and classic signs of respirator...",RSV bronchiolitis,✔️ [0.000]



Baseline exact-match accuracy: 0.00%
Example 1: gold=Eye Condition | pred=Multiple Sclerosis | correct=False
Example 2: gold=COPD/Respiratory | pred=Aspiration pneumonia | correct=False
Example 3: gold=Cardiovascular | pred=Long QT syndrome | correct=False
Example 4: gold=Arthritis | pred=Acute gout (acute gouty arthritis) | correct=False
Example 5: gold=Cardiovascular | pred=Orthostatic hypotension | correct=False


In [8]:
# ============================================================
# 6) METRIC WITH FEEDBACK (GEPA-READY)
# ============================================================
def metric_with_feedback(example, pred, trace=None, pred_name=None, pred_trace=None):
    fb_valid, s_valid = feedback_valid_label(pred.medical_condition)
    fb_acc, s_acc = feedback_accuracy(example, pred.medical_condition)

    total = (0.10 * s_valid) + (0.90 * s_acc)

    if pred_name is None:
        return total

    if pred_name == "generator.predict":
        feedback = (
            f"Label Validity Feedback:\n{fb_valid}\n\n"
            f"Accuracy Feedback:\n{fb_acc}\n\n"
            f"Allowed Labels:\n{ALLOWED_LABELS_TEXT}\n\n"
            f"Overall Score: {total:.3f}\n"
            "Think step-by-step about the organ system, hallmark symptoms, chronic disease context, and the single best label to return."
        )
    else:
        feedback = "No specific predictor matched. Review the classification reasoning carefully."

    return dspy.Prediction(score=total, feedback=feedback)

In [9]:
# ============================================================
# 7) GEPA OPTIMIZER
# ============================================================
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=8,
    track_stats=True,
    track_best_outputs=True,
    use_merge=False,
    reflection_lm=reflection_lm,
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/04/01 21:10:03 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 500 metric calls of the program. This amounts to 2.91 full evals on the train+val set.
2026/04/01 21:10:03 INFO dspy.teleprompt.gepa.gepa: Using 30 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/500 [00:00<?, ?rollouts/s]2026/04/01 21:10:19 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 30 (0.0%)
2026/04/01 21:10:19 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.0 over 30 / 30 examples
GEPA Optimization:   6%|▌         | 30/500 [00:15<04:02,  1.94rollouts/s]2026/04/01 21:10:19 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.0


Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:03<00:00,  1.07s/it]

2026/04/01 21:10:22 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)


2026/04/01 21:10:30 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for generator.predict: # Task: Single‑Label Medical Condition Classification

**Input**  
You will receive a single text field named `patient_notes`.  
It contains a free‑form clinical narrative describing a patient's history, symptoms, examination findings, labs, and any relevant contextual information.

**Goal**  
Read the notes carefully and determine **the single best medical‑condition category** that best captures the patient’s presenting problem.  

The answer must be **exactly one** of the following *allowed* labels (no other labels, no disease names, no extra text):

```
Arthritis
COPD/Respiratory
Cardiovascular
Diabetes
Eye Condition
Neurological
```

**Output format**  
Your response should contain two parts:

```
### reasoning
[write a concise, step‑by‑step justification for your chosen label.  Highlight key symptom patterns, the dominant organ system involved, and any contextual clues (e.g., 

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:02<00:00,  1.02it/s]

...


Average Metric: 2.10 / 3 (70.0%): 100%|██████████| 3/3 [00:02<00:00,  1.41it/s] 

2026/04/01 21:20:29 INFO dspy.evaluate.evaluate: Average Metric: 2.1 / 3 (70.0%)


2026/04/01 21:20:38 INFO dspy.teleprompt.gepa.gepa: Iteration 40: Proposed new text for generator.predict: ## 🎯  Task Overview

Given a single block of patient notes (symptoms, history, exam findings, and test results), the assistant must:
1. **Identify the main organ system** described in the notes.  
2. **Match the clues** to one of six fixed medical categories (see table below).  
3. **Produce a concise 1‑3 sentence reasoning** that:  
   * contains only one idea per sentence,  
   * references only clinical clues or exam/test hallmarks,  
   * never names a specific disease, drug, or therapy.  
4. **Output the single category name** from the list below.  

The assistant must deliver the output in the exact format shown in the “Output format” section. Any deviation (extra text, missing headings, disease names, etc.) will be considered an error.

---

## 📑 Input format

The input will always contain a single key named `patient_notes`. Its value is a free‑text paragraph or list of obs

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:02<00:00,  1.01it/s]

2026/04/01 21:20:55 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/04/01 21:20:55 INFO dspy.teleprompt.gepa.gepa: Iteration 41: All subsample scores perfect. Skipping.
2026/04/01 21:20:55 INFO dspy.teleprompt.gepa.gepa: Iteration 41: Reflective mutation did not propose a new candidate
GEPA Optimization: 100%|█████████▉| 498/500 [10:52<00:02,  1.31s/rollouts]

In [10]:
# ============================================================
# 8) SEE OPTIMIZED PROMPTS
# ============================================================
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: generator.predict
Prompt:
You are provided with a block of patient notes that describe symptoms, history, exam findings, and test results.  
Your job is to assign one of six possible medical categories **without naming any specific disease**.  

**Output format (must match exactly):**  
```
### reasoning
<1–3 concise sentences, each expressing a single idea>
 
### medical_condition
<one of: Arthritis, COPD/Respiratory, Cardiovascular, Diabetes, Eye Condition, Neurological>
*********************************


In [11]:
# ============================================================
# 9) FINAL POST-GEPA EVALUATION
# ============================================================
print("\n== GEPA-Optimized Evaluation ==")
evaluate(optimized_program)

optimized_accuracy, optimized_rows = dataset_accuracy(optimized_program, test_set)
print(f"\nGEPA exact-match accuracy: {optimized_accuracy:.2%}")
print(f"Accuracy lift: {(optimized_accuracy - baseline_accuracy):+.2%}")
print_sample_predictions(optimized_rows)


== GEPA-Optimized Evaluation ==
Average Metric: 24.40 / 30 (81.3%): 100%|██████████| 30/30 [00:31<00:00,  1.05s/it]

2026/04/01 21:21:27 INFO dspy.evaluate.evaluate: Average Metric: 24.4 / 30 (81.3%)


,patient_notes,gold_label,reasoning,medical_condition,metric
0,12-year-old female whose parents noticed progressive pallor of her...,Eye Condition,The presence of pallor of the optic discs and severely reduced vis...,Eye Condition,✔️ [1.000]
1,"A 78-year-old female, recently discharged from the hospital after ...",COPD/Respiratory,The patient's recent stroke and dysphagia are significant risk fac...,COPD/Respiratory,✔️ [1.000]
2,19-year-old female presents after an episode of syncope while swim...,Cardiovascular,The occurrence of syncope during exertion (swimming) in a young in...,Cardiovascular,✔️ [1.000]
3,A 75-year-old male with a history of chronic kidney disease and di...,Arthritis,While the polyarticular involvement might initially suggest rheuma...,Arthritis,✔️ [1.000]
4,"81-year-old female admitted after a fall at home, sustaining a hip...",Cardiovascular,"The patient's recurrent lightheadedness, dizziness, and syncope up...",Cardiovascular,✔️ [1.000]
5,"An 85-year-old male, residing in a skilled nursing facility, prese...",COPD/Respiratory,"The patient's advanced age, institutionalization, acute onset of f...",COPD/Respiratory,✔️ [1.000]
6,A 58-year-old obese male (BMI 38 kg/m²) presents with chronic dayt...,COPD/Respiratory,While ankle edema and dyspnea might suggest primary cardiac or ren...,### medical_condition\nCardiovascular,✔️ [0.000]
7,92-year-old female found unresponsive at home. Paramedics report p...,Cardiovascular,"The presentation of sudden collapse, profound hypotension, a pulsa...",Cardiovascular,✔️ [1.000]
8,"A 72-year-old male, with a long-standing history of tobacco abuse ...",COPD/Respiratory,"The patient's advanced age, extensive smoking history, chronic pro...",COPD/Respiratory,✔️ [1.000]
9,A 9-month-old male infant presents with a 3-day history of rhinorr...,COPD/Respiratory,"The patient's age, viral prodrome, and classic signs of respirator...",COPD/Respiratory,✔️ [1.000]



GEPA exact-match accuracy: 80.00%
Accuracy lift: +80.00%
Example 1: gold=Eye Condition | pred=Eye Condition | correct=True
Example 2: gold=COPD/Respiratory | pred=COPD/Respiratory | correct=True
Example 3: gold=Cardiovascular | pred=Cardiovascular | correct=True
Example 4: gold=Arthritis | pred=Arthritis | correct=True
Example 5: gold=Cardiovascular | pred=Cardiovascular | correct=True


## Performance Improvement Summary

After running the notebook, fill in the baseline accuracy, optimized accuracy, and absolute lift on the held-out test split.

| Metric | Value |
|--------|-------|
| Baseline exact-match accuracy | `TBD` |
| GEPA exact-match accuracy | `TBD` |
| Accuracy lift | `TBD` |